# Run 12a — Baseline on Full Dataset (4451 segments)
Same pipeline that scored 0.979 on 632 segments. NOAA/BirdNET disabled.  
Run cells top-to-bottom. If a cell crashes, fix and re-run just that cell.

In [ ]:
# Cell 1: Imports & Config
import os, re, time, json, gc
import numpy as np
import librosa
from prepare import (
    TARGET_SR, SEGMENT_SECONDS, SEGMENT_SAMPLES,
    N_FFT, HOP_LENGTH, N_MELS, F_MIN, F_MAX,
    TIME_BUDGET, CACHE_DIR, RESULTS_DIR, DEVICE,
    BAND_LOW, BAND_MID, BAND_HIGH,
    find_wav_files, load_audio, segment_audio, highpass_filter,
    compute_melspec, compute_band_power, compute_rms,
    evaluate_clustering, evaluate_discovery,
    build_dataset,
)

# Hyperparameters
N_MFCC = 20
USE_DELTA_MFCC = True
USE_BAND_POWER = True
USE_SPECTRAL = True
USE_SPECTRAL_CONTRAST = True
USE_ZCR = True
USE_RMS = True

REDUCER = "umap"
N_COMPONENTS = 20
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.0
UMAP_METRIC = "euclidean"

CLUSTERER = "hdbscan"
HDBSCAN_MIN_CLUSTER = 10
HDBSCAN_MIN_SAMPLES = 5

print(f"Device: {DEVICE}")
print("Config loaded.")

In [ ]:
# Cell 2: Feature Extraction (with caching)
# ~20 min on g4dn first run, instant on re-run
t_start = time.time()

cache_features_path = os.path.join(CACHE_DIR, "tier1_features.npy")
cache_mels_path = os.path.join(CACHE_DIR, "mels.npy")
cache_meta_path = os.path.join(CACHE_DIR, "metadata.json")

recordings = find_wav_files()
print(f"Found {len(recordings)} recordings")

cache_hit = False
if os.path.exists(cache_features_path) and os.path.exists(cache_mels_path) and os.path.exists(cache_meta_path):
    print("Loading cached features...")
    try:
        features = np.load(cache_features_path)
        all_mels_array = np.load(cache_mels_path)
        with open(cache_meta_path, "r") as f:
            metadata = json.load(f)
        print(f"Cache loaded: {len(metadata)} segments, features={features.shape}, mels={all_mels_array.shape}")
        cache_hit = True
    except Exception as e:
        print(f"Cache load failed: {e}, re-extracting...")

if not cache_hit:
    print(f"Extracting features from {len(recordings)} recordings...")
    all_features = []
    all_mels_list = []
    metadata = []

    for ri, rec in enumerate(recordings):
        print(f"  [{ri+1}/{len(recordings)}] {rec['unit']}/{rec['filename']} ({rec['duration_s']:.0f}s @ {rec['sample_rate']}Hz)")
        audio, sr = load_audio(rec["path"])
        audio = highpass_filter(audio, sr)
        file_segments = segment_audio(audio, sr)
        del audio

        for i, seg in enumerate(file_segments):
            feat_vec = []
            mel = compute_melspec(seg)
            all_mels_list.append(mel)
            mfcc = librosa.feature.mfcc(y=seg, sr=48000, n_mfcc=N_MFCC)
            feat_vec.extend(mfcc.mean(axis=1))
            feat_vec.extend(mfcc.std(axis=1))
            if USE_DELTA_MFCC:
                d1 = librosa.feature.delta(mfcc)
                feat_vec.extend(d1.mean(axis=1))
                feat_vec.extend(d1.std(axis=1))
            if USE_BAND_POWER:
                bp = compute_band_power(seg)
                feat_vec.extend([bp["low"], bp["mid"], bp["high"]])
            if USE_SPECTRAL:
                sc = librosa.feature.spectral_centroid(y=seg, sr=48000)[0]
                sb = librosa.feature.spectral_bandwidth(y=seg, sr=48000)[0]
                sr_feat = librosa.feature.spectral_rolloff(y=seg, sr=48000)[0]
                sf_feat = librosa.feature.spectral_flatness(y=seg)[0]
                for x in [sc, sb, sr_feat, sf_feat]:
                    feat_vec.extend([x.mean(), x.std()])
            if USE_SPECTRAL_CONTRAST:
                scon = librosa.feature.spectral_contrast(y=seg, sr=48000)
                feat_vec.extend(scon.mean(axis=1))
                feat_vec.extend(scon.std(axis=1))
            if USE_ZCR:
                z = librosa.feature.zero_crossing_rate(seg)[0]
                feat_vec.extend([z.mean(), z.std()])
            if USE_RMS:
                r = librosa.feature.rms(y=seg)[0]
                feat_vec.extend([r.mean(), r.std()])
            mel_stats = mel
            feat_vec.extend([mel_stats.mean(), mel_stats.std(),
                            mel_stats.max(), np.percentile(mel_stats, 90)])
            all_features.append(feat_vec)
            metadata.append({
                "file": rec["filename"], "unit": rec["unit"],
                "segment_idx": i, "offset_s": i * 10,
                "path": rec["path"],
            })
        del file_segments
        gc.collect()

    features = np.array(all_features, dtype=np.float32)
    del all_features; gc.collect()

    time_dim = min(m.shape[1] for m in all_mels_list)
    all_mels_array = np.array([m[:, :time_dim] for m in all_mels_list], dtype=np.float32)
    all_mels_array = (all_mels_array - all_mels_array.min()) / (all_mels_array.max() - all_mels_array.min() + 1e-8)
    del all_mels_list; gc.collect()

    # Save cache
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.save(cache_features_path, features)
    np.save(cache_mels_path, all_mels_array)
    with open(cache_meta_path, "w") as f:
        json.dump(metadata, f)
    print(f"Cache saved.")

n_segs = len(metadata)
t_extract = time.time() - t_start
print(f"\nTotal segments: {n_segs}")
print(f"Features: {features.shape}")
print(f"Mels: {all_mels_array.shape}")
print(f"Extraction time: {t_extract:.1f}s")

In [ ]:
# Cell 3: UMAP + HDBSCAN (Tier 1 baseline)
from sklearn.preprocessing import StandardScaler
import umap
import hdbscan as hdbscan_lib

def reduce_dimensions(feats):
    scaled = StandardScaler().fit_transform(feats)
    return umap.UMAP(
        n_components=N_COMPONENTS, n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST, metric=UMAP_METRIC, random_state=42,
    ).fit_transform(scaled)

def cluster_hdbscan(feats_reduced):
    return hdbscan_lib.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER,
        min_samples=HDBSCAN_MIN_SAMPLES,
    ).fit_predict(feats_reduced)

print("--- Tier 1: UMAP + HDBSCAN ---")
t1 = time.time()
features_reduced = reduce_dimensions(features)
print(f"UMAP: {features_reduced.shape}, {time.time()-t1:.1f}s")

t1 = time.time()
labels = cluster_hdbscan(features_reduced)
print(f"HDBSCAN: {time.time()-t1:.1f}s")

eval_result = evaluate_clustering(labels, features_reduced, method_name="tier1")
best_eval = eval_result
best_labels = labels
best_reduced = features_reduced
best_name = "tier1"

n_clusters = len(set(labels[labels >= 0]))
n_noise = (labels == -1).sum()
coverage = 1.0 - n_noise / len(labels)
print(f"\nTier 1 baseline:")
print(f"  composite_score: {eval_result['composite_score']:.6f}")
print(f"  silhouette:      {eval_result['silhouette']:.6f}")
print(f"  n_clusters:      {n_clusters}")
print(f"  n_noise:         {n_noise} ({100*(1-coverage):.1f}%)")
print(f"  coverage:        {coverage:.4f}")

In [ ]:
# Cell 4: CNN Classifier (Tier 3a)
# ~5 min on g4dn GPU
import torch
import torch.nn as nn

def spec_augment(batch, freq_mask_param=20, time_mask_param=30):
    b, c, f, t = batch.shape
    augmented = batch.clone()
    for idx in range(b):
        f_start = torch.randint(0, max(1, f - freq_mask_param), (1,)).item()
        f_width = torch.randint(0, freq_mask_param + 1, (1,)).item()
        augmented[idx, :, f_start:f_start+f_width, :] = 0
        t_start = torch.randint(0, max(1, t - time_mask_param), (1,)).item()
        t_width = torch.randint(0, time_mask_param + 1, (1,)).item()
        augmented[idx, :, :, t_start:t_start+t_width] = 0
    return augmented

def train_cnn_classifier(mels, pseudo_labels):
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"  CNN device: {device}")

    mask = pseudo_labels >= 0
    labeled_indices = np.where(mask)[0]
    n_classes = len(set(pseudo_labels[mask]))
    print(f"  Training CNN: {len(labeled_indices)} labeled segments, {n_classes} classes")

    X_all = mels
    X = X_all[labeled_indices]
    y = pseudo_labels[mask]
    time_dim = X_all.shape[2]

    class MarineCNN(nn.Module):
        def __init__(self, n_mels, time_dim, n_classes, feat_dim=64):
            super().__init__()
            self.conv = nn.Sequential(
                nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(), nn.BatchNorm2d(16),
                nn.Dropout2d(0.1),
                nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
                nn.Dropout2d(0.1),
                nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
                nn.Dropout2d(0.15),
                nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
                nn.Dropout2d(0.2),
            )
            dummy = torch.zeros(1, 1, n_mels, time_dim)
            conv_out = self.conv(dummy)
            flat_dim = conv_out.numel()
            self.features = nn.Sequential(
                nn.Flatten(), nn.Linear(flat_dim, feat_dim), nn.ReLU(), nn.Dropout(0.3),
            )
            self.classifier = nn.Linear(feat_dim, n_classes)
        def forward(self, x):
            h = self.conv(x)
            feats = self.features(h)
            logits = self.classifier(feats)
            return logits, feats

    model = MarineCNN(128, time_dim, n_classes, feat_dim=64).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-3)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    X_tensor = torch.from_numpy(X[:, np.newaxis, :, :])
    y_tensor = torch.from_numpy(y.astype(np.int64))
    batch_size = 32
    n = len(X_tensor)
    n_epochs = 500

    warmup_epochs = 20
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / warmup_epochs
        return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (n_epochs - warmup_epochs)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    model.train()
    for epoch in range(n_epochs):
        perm = torch.randperm(n)
        total_loss = 0; correct = 0; total = 0
        for i in range(0, n, batch_size):
            batch_x = X_tensor[perm[i:i+batch_size]].to(device)
            batch_y = y_tensor[perm[i:i+batch_size]].to(device)
            batch_x = spec_augment(batch_x)
            logits, _ = model(batch_x)
            loss = criterion(logits, batch_y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
            correct += (logits.argmax(1) == batch_y).sum().item()
            total += len(batch_y)
        scheduler.step()
        if (epoch + 1) % 100 == 0:
            print(f"    Epoch {epoch+1}/{n_epochs}: loss={total_loss:.4f}, acc={correct/total*100:.1f}%")

    model.eval()
    X_all_tensor = torch.from_numpy(X_all[:, np.newaxis, :, :])
    feats_list = []
    with torch.no_grad():
        for i in range(0, len(X_all_tensor), batch_size):
            batch = X_all_tensor[i:i+batch_size].to(device)
            _, feats = model(batch)
            feats_list.append(feats.cpu().numpy())
    learned = np.vstack(feats_list)
    print(f"  CNN features: {learned.shape}")
    return learned

print("--- Tier 3a: CNN Classifier ---")
t1 = time.time()
cnn_features = train_cnn_classifier(all_mels_array, labels)
cnn_norm = StandardScaler().fit_transform(cnn_features)
tier1_norm = StandardScaler().fit_transform(features)

# CNN-only
fr_cnn = reduce_dimensions(cnn_norm)
lb_cnn = cluster_hdbscan(fr_cnn)
ev_cnn = evaluate_clustering(lb_cnn, fr_cnn, method_name="tier3_cnn")

# CNN + Tier1
combined = np.hstack([tier1_norm, cnn_norm])
fr_comb = reduce_dimensions(combined)
lb_comb = cluster_hdbscan(fr_comb)
ev_comb = evaluate_clustering(lb_comb, fr_comb, method_name="tier3_combined")

print(f"  CNN-only: {ev_cnn['composite_score']:.6f}, Combined: {ev_comb['composite_score']:.6f}")
print(f"  CNN time: {time.time()-t1:.1f}s")

for ev, lb, fr, name in [(ev_cnn, lb_cnn, fr_cnn, "cnn-only"), (ev_comb, lb_comb, fr_comb, "cnn+tier1")]:
    if ev['composite_score'] > best_eval['composite_score']:
        print(f"  ** {name} IMPROVED: {ev['composite_score']:.6f} **")
        best_eval, best_labels, best_reduced, best_name = ev, lb, fr, name

print(f"\nBest so far: {best_name} = {best_eval['composite_score']:.6f}")

In [ ]:
# Cell 5: Contrastive Learning (Tier 3b)
# ~8 min on g4dn GPU
import torch.nn.functional as F

def train_contrastive(mels, temperature=0.1, feat_dim=64, n_epochs=300):
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"  Contrastive device: {device}")

    X_all = mels
    time_dim = X_all.shape[2]

    class Encoder(nn.Module):
        def __init__(self, n_mels, time_dim, feat_dim):
            super().__init__()
            self.conv = nn.Sequential(
                nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(), nn.BatchNorm2d(16),
                nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
                nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
                nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
            )
            dummy = torch.zeros(1, 1, n_mels, time_dim)
            conv_out = self.conv(dummy)
            flat_dim = conv_out.numel()
            self.features = nn.Sequential(nn.Flatten(), nn.Linear(flat_dim, feat_dim), nn.ReLU())
            self.projector = nn.Sequential(nn.Linear(feat_dim, feat_dim), nn.ReLU(), nn.Linear(feat_dim, feat_dim))
        def forward(self, x, return_projection=False):
            h = self.conv(x)
            feats = self.features(h)
            if return_projection:
                proj = self.projector(feats)
                return feats, F.normalize(proj, dim=1)
            return feats

    model = Encoder(128, time_dim, feat_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-3)

    X_tensor = torch.from_numpy(X_all[:, np.newaxis, :, :])
    n = len(X_tensor)
    batch_size = 64

    warmup_epochs = 15
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / warmup_epochs
        return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (n_epochs - warmup_epochs)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    print(f"  Training contrastive: {n} segments, {n_epochs} epochs")
    model.train()
    for epoch in range(n_epochs):
        perm = torch.randperm(n)
        total_loss = 0; n_batches = 0
        for i in range(0, n, batch_size):
            batch_x = X_tensor[perm[i:i+batch_size]].to(device)
            if len(batch_x) < 4: continue
            view1 = spec_augment(batch_x, freq_mask_param=25, time_mask_param=35)
            view2 = spec_augment(batch_x, freq_mask_param=25, time_mask_param=35)
            _, z1 = model(view1, return_projection=True)
            _, z2 = model(view2, return_projection=True)
            bsz = z1.shape[0]
            z = torch.cat([z1, z2], dim=0)
            sim = torch.mm(z, z.t()) / temperature
            mask = torch.eye(2 * bsz, dtype=torch.bool, device=device)
            sim.masked_fill_(mask, -1e9)
            cl_labels = torch.cat([torch.arange(bsz, 2*bsz, device=device), torch.arange(0, bsz, device=device)])
            loss = F.cross_entropy(sim, cl_labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item(); n_batches += 1
        scheduler.step()
        if (epoch + 1) % 50 == 0:
            print(f"    Epoch {epoch+1}/{n_epochs}: loss={total_loss/max(n_batches,1):.4f}")

    model.eval()
    feats_list = []
    with torch.no_grad():
        for i in range(0, n, batch_size):
            batch = X_tensor[i:i+batch_size].to(device)
            feats = model(batch, return_projection=False)
            feats_list.append(feats.cpu().numpy())
    learned = np.vstack(feats_list)
    print(f"  Contrastive features: {learned.shape}")
    return learned

print("--- Tier 3b: Contrastive Learning (SimCLR) ---")
t1 = time.time()
cl_features = train_contrastive(all_mels_array)
cl_norm = StandardScaler().fit_transform(cl_features)

# CL-only
fr_cl = reduce_dimensions(cl_norm)
lb_cl = cluster_hdbscan(fr_cl)
ev_cl = evaluate_clustering(lb_cl, fr_cl, method_name="tier3_contrastive")

# CL + Tier1
fr_clt1 = reduce_dimensions(np.hstack([tier1_norm, cl_norm]))
lb_clt1 = cluster_hdbscan(fr_clt1)
ev_clt1 = evaluate_clustering(lb_clt1, fr_clt1, method_name="tier3_cl_combined")

# CNN + CL
fr_cnncl = reduce_dimensions(np.hstack([cnn_norm, cl_norm]))
lb_cnncl = cluster_hdbscan(fr_cnncl)
ev_cnncl = evaluate_clustering(lb_cnncl, fr_cnncl, method_name="tier3_cnn_cl")

print(f"  CL-only: {ev_cl['composite_score']:.6f}, CL+T1: {ev_clt1['composite_score']:.6f}, CNN+CL: {ev_cnncl['composite_score']:.6f}")
print(f"  Contrastive time: {time.time()-t1:.1f}s")

for ev, lb, fr, name in [
    (ev_cl, lb_cl, fr_cl, "contrastive-only"),
    (ev_clt1, lb_clt1, fr_clt1, "contrastive+tier1"),
    (ev_cnncl, lb_cnncl, fr_cnncl, "cnn+contrastive"),
]:
    if ev['composite_score'] > best_eval['composite_score']:
        print(f"  ** {name} IMPROVED: {ev['composite_score']:.6f} **")
        best_eval, best_labels, best_reduced, best_name = ev, lb, fr, name

print(f"\nBest so far: {best_name} = {best_eval['composite_score']:.6f}")

In [ ]:
# Cell 6: Kitchen Sink (T1 + CNN + CL) + Save Results
print("--- Kitchen Sink ---")
t1 = time.time()
combined_sink = np.hstack([tier1_norm, cnn_norm, cl_norm])
fr_sink = reduce_dimensions(combined_sink)
lb_sink = cluster_hdbscan(fr_sink)
ev_sink = evaluate_clustering(lb_sink, fr_sink, method_name="kitchen_sink_T1+CNN+CL")
print(f"  T1+CNN+CL: {ev_sink['composite_score']:.6f} ({combined_sink.shape[1]} dims)")
print(f"  Kitchen sink time: {time.time()-t1:.1f}s")

if ev_sink['composite_score'] > best_eval['composite_score']:
    print(f"  ** Kitchen sink IMPROVED: {ev_sink['composite_score']:.6f} **")
    best_eval, best_labels, best_reduced, best_name = ev_sink, lb_sink, fr_sink, "kitchen_sink"

# Discovery
discovery = evaluate_discovery(best_labels, metadata, best_reduced, None)

# Save result
os.makedirs(RESULTS_DIR, exist_ok=True)
timestamp = time.strftime("%Y%m%d_%H%M%S")
result_path = os.path.join(RESULTS_DIR, f"result_{timestamp}.json")
result_data = {
    "config": {
        "run": "12a", "n_mfcc": N_MFCC, "reducer": REDUCER,
        "n_components": N_COMPONENTS, "clusterer": CLUSTERER,
        "hdbscan_min_cluster": HDBSCAN_MIN_CLUSTER,
        "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
    },
    "evaluation": best_eval,
    "discovery": discovery,
    "best_method": best_name,
}
with open(result_path, "w") as f:
    json.dump(result_data, f, indent=2)

# Final summary
print(f"\n{'='*60}")
print(f"RUN 12a RESULTS ({best_name})")
print(f"{'='*60}")
print(f"composite_score:  {best_eval['composite_score']:.6f}")
print(f"silhouette:       {best_eval['silhouette']:.6f}")
print(f"calinski_harabasz:{best_eval['calinski_harabasz']:.6f}")
print(f"n_clusters:       {best_eval['n_clusters']}")
print(f"n_noise:          {best_eval['n_noise']}")
print(f"coverage:         {best_eval['coverage']:.4f}")
print(f"total_segments:   {n_segs}")
print(f"saved:            {result_path}")
print(f"\n632-segment baseline was 0.979342 — compare above.")